# Warped toy regression (Damianou & Lawrence 2013, §4.1 / fig. 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/PyDeepGP/blob/main/examples/damianou2013/notebooks/02_toy_warped_regression.ipynb)

**Paper:** [Damianou, A. and Lawrence, N. D. (2013). *Deep Gaussian Processes*. In *AISTATS* (pp. 207–215).](https://proceedings.mlr.press/v31/damianou13a.html)

A non-stationary regression problem is created by stacking Linear+RBF → RBF GPs. Only 25 of 120 points are observed. A standard sparse GP is compared to a DeepGP with **two** hidden layers (one more than the generative depth). The paper reports lower test MSE for the deep model across repeats (fig. 3b).


In [ ]:
# Install dependencies when running on Google Colab
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    pkgs = ["numpy>=1.7", "scipy>=0.12", "matplotlib", "GPy>=1.15"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/SheffieldML/PyDeepGP.git@main",
    ])
    print("Installed GPy and PyDeepGP for Colab.")
else:
    print("Local environment — ensure PyDeepGP and GPy are importable.")


In [ ]:
# Set QUICK=True for a short smoke run (fewer iterations / smaller models).
# Set QUICK=False for a paper-scale demonstration (slower).
QUICK = True
SEED = 42

REPEATS = 2 if QUICK else 10


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import GPy
import deepgp

np.random.seed(SEED)

def sample_gp_prior(X, kernel, output_dim, noise=1e-6):
    K = kernel.K(X) + noise * np.eye(X.shape[0])
    return np.linalg.cholesky(K) @ np.random.randn(X.shape[0], output_dim)

def rbf_bias(q, ard=True):
    return GPy.kern.RBF(q, ARD=ard) + GPy.kern.Bias(q)

def generate_warped(n=120, seed=0):
    rng = np.random.RandomState(seed)
    np.random.seed(seed)
    X = np.linspace(-1.0, 1.0, n)[:, None]
    H = sample_gp_prior(X, GPy.kern.Linear(1) + GPy.kern.RBF(1, lengthscale=0.3), 1)
    H += 0.01 * rng.randn(*H.shape)
    Y = sample_gp_prior(H, GPy.kern.RBF(1, lengthscale=0.25), 10)
    Y += 0.05 * rng.randn(*Y.shape)
    return X, H, Y

def mse(a, b):
    return float(np.mean((np.asarray(a) - np.asarray(b)) ** 2))


## One draw of the warped process (fig. 3a style)


In [ ]:
X, H, Y = generate_warped(seed=SEED)
fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
axes[0].plot(X[:, 0], H[:, 0], "k-")
axes[0].set_ylabel("hidden warp")
axes[0].set_title("Warped generative process")
axes[1].plot(X[:, 0], Y[:, 0], label="Y[:,0]")
axes[1].plot(X[:, 0], Y[:, 1], label="Y[:,1]")
axes[1].set_xlabel("X")
axes[1].set_ylabel("outputs")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()


## GP vs DeepGP over repeats

Training size 25 (paper); DeepGP has two hidden layers between $X$ and $Y$.


In [ ]:
from GPy.core.parameterization.variational import NormalPosterior, VariationalPosterior

def out_var(y):
    if isinstance(y, (NormalPosterior, VariationalPosterior)):
        return float(np.asarray(y.mean).var())
    return float(np.asarray(y).var())

def fit_gp(Xtr, Ytr, M):
    m = GPy.models.SparseGPRegression(
        Xtr, Ytr, kernel=GPy.kern.RBF(1) + GPy.kern.Bias(1), num_inducing=M
    )
    m.Gaussian_noise.variance = Ytr.var() * 0.01
    m.Gaussian_noise.variance.fix()
    m.optimize(max_iters=100, messages=False)
    m.Gaussian_noise.variance.unfix()
    m.optimize(max_iters=400, messages=False)
    return m

def fit_dgp(Xtr, Ytr, M, q1, q2, it_f, it_u):
    m = deepgp.DeepGP(
        [Ytr.shape[1], q1, q2, Xtr.shape[1]],
        Ytr,
        X=Xtr,
        kernels=[rbf_bias(q1), rbf_bias(q2), rbf_bias(Xtr.shape[1], ard=False)],
        num_inducing=M,
        back_constraint=False,
    )
    for layer in m.layers:
        layer.Gaussian_noise.variance = out_var(layer.Y) * 0.01
        layer.Gaussian_noise.variance.fix()
    m.optimize(max_iters=it_f, messages=False)
    for layer in m.layers:
        layer.Gaussian_noise.variance.unfix()
    m.optimize(max_iters=it_u, messages=False)
    return m

n_train = 20 if QUICK else 25
q1, q2 = (3, 2) if QUICK else (4, 3)
M = 12 if QUICK else 20
it_f, it_u = (40, 120) if QUICK else (200, 1200)

gp_mses, dgp_mses = [], []
for r in range(REPEATS):
    seed = SEED + r
    X, _, Y = generate_warped(seed=seed)
    rng = np.random.RandomState(seed + 1000)
    idx = rng.permutation(X.shape[0])
    tr, te = np.sort(idx[:n_train]), np.sort(idx[n_train:])
    Xtr, Ytr, Xte, Yte = X[tr], Y[tr], X[te], Y[te]
    m_gp = fit_gp(Xtr, Ytr, M)
    m_dgp = fit_dgp(Xtr, Ytr, M, q1, q2, it_f, it_u)
    gp_mses.append(mse(m_gp.predict(Xte)[0], Yte))
    dgp_mses.append(mse(m_dgp.predict(Xte)[0], Yte))
    print(f"Repeat {r+1}/{REPEATS}: GP={gp_mses[-1]:.4f}  DeepGP={dgp_mses[-1]:.4f}")

print(f"Mean MSE (±std): GP {np.mean(gp_mses):.4f}±{np.std(gp_mses):.4f}  "
      f"DeepGP {np.mean(dgp_mses):.4f}±{np.std(dgp_mses):.4f}")

fig, ax = plt.subplots(figsize=(5.5, 3.5))
xs = np.arange(1, REPEATS + 1)
ax.plot(xs, gp_mses, "o-", label="GP")
ax.plot(xs, dgp_mses, "s-", label="DeepGP")
ax.set_xlabel("experiment #")
ax.set_ylabel("test MSE")
ax.set_title("Fig. 3(b) style — MSE by repeat")
ax.legend()
plt.show()


## References

- Damianou, A. and Lawrence, N. D. (2013). *Deep Gaussian Processes*. In *AISTATS* (pp. 207–215). — [PDF](https://proceedings.mlr.press/v31/damianou13a.html)
- Script counterpart: `examples/damianou2013/toy_warped_regression.py`
